# Stage 3: speed trial for the main run

Notebook 05 proved that training works and resumes. It also showed the main run is too slow at micro batch 2 with gradient checkpointing (582 tokens/s, projected 14.9 h for 950 steps). This notebook tries four settings for a few real steps each and tells you which is fastest and still fits in GPU memory:

| setting | micro batch | gradient checkpointing |
| --- | --- | --- |
| `mb2_ckpt_on` | 2 | on (the smoke-run setting) |
| `mb4_ckpt_on` | 4 | on |
| `mb1_ckpt_off` | 1 | off |
| `mb2_ckpt_off` | 2 | off |

Each runs in its own process, so an out-of-memory error in one only marks that setting as failed. Every optimizer step still sees 32 blocks, so the winner does not change what is learned, only how fast.

**Send back `speed_report.json`** (written to the notebook's working folder), then put the winning `MICRO_BATCH` and `GRAD_CHECKPOINT` into notebook 07.

Settings before running: Accelerator **GPU T4 x2** (one is used), Internet **on**, Input **Add Input > Notebooks** with the finished output of `03_extend_export_qwen3`. A Hub repo and the `HF_TOKEN` secret are optional: with them the notebook also proves checkpoint and milestone upload and download work. About 20 minutes including data preparation.

In [ ]:
# Speed and memory are measured without the Hub. If you set HUB_REPO (a PRIVATE repo you made), the
# notebook also tests the checkpoint and milestone upload and download on a throwaway "<repo>-test" repo.
HUB_REPO = "hashiraziz21/zubaan-qwen3-ckpt"            # e.g. "your-name/zubaan-qwen3-ckpt"; empty skips the Hub test
MICRO_BATCH = 2          # used by the in-process 5-step check only
GRAD_CHECKPOINT = True
PHASE_A_STEPS = 100      # unused here; the config cell shares its code with notebook 07
PHASE_B_STEPS = 700
OUT_NAME = "adapt_speed"
HUB = None
MODE = "check"           # the shared check cell reads this name

In [ ]:
# Install zubaan at one exact commit. Do not edit the next line except to change the commit.
COMMIT = "98fb1a5"
REPO = f"git+https://github.com/Hashir-Aziz123/zubaan.git@{COMMIT}"
!pip install -q "zubaan[audit,train] @ {REPO}"
# Kaggle may already hold an older copy with the same version number, which pip keeps.
# Force our commit (dependencies are already in place).
!pip install -q --force-reinstall --no-deps "zubaan @ {REPO}"
# Kaggle ships torchao 0.10, which peft refuses ("only versions above 0.16 are supported") when it
# builds LoRA layers. We never use torchao, so remove it; peft then skips its torchao support.
!pip uninstall -y -q torchao
!zubaan version

In [ ]:
# CHECK 1: the installed code is the commit above, not a cached older copy.
# If this fails, use Run > Restart session and run the cells again from the top.
import json
from importlib import metadata

direct = json.loads(metadata.distribution("zubaan").read_text("direct_url.json"))
installed = direct["vcs_info"]["commit_id"]
print("installed commit:", installed)
assert installed.startswith(COMMIT), f"installed {installed}, wanted {COMMIT}"

import zubaan.adapt.pipeline  # noqa: F401

print("OK: code version verified")

In [ ]:
import os
import subprocess

# Quiet progress bars: they can overwhelm the notebook front end.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# CHECK 2: a GPU with about 16 GB. Training uses one GPU (fp16: a T4 and a P100 have no BF16).
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
except FileNotFoundError:
    raise SystemExit("No GPU in this session (nvidia-smi is missing). Settings > Accelerator > GPU T4 x2, then run again.") from None
assert r.returncode == 0, "no GPU found: turn on a GPU accelerator in the notebook settings"
print(r.stdout)
gpu_name, cap, mem = [x.strip() for x in r.stdout.splitlines()[0].split(",")]
assert float(mem.split()[0]) >= 14000, f"{gpu_name} has {mem}; need about 16 GB"
if float(cap) < 7.0:
    print("WARNING: compute capability", cap, "has no fp16 tensor cores; this will be slow. A T4 is better.")

import peft
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "torch cannot see the GPU"
print("OK: GPU", gpu_name, "compute capability", cap, mem)

In [ ]:
# The Hub token is a Kaggle secret named HF_TOKEN (Add-ons > Secrets). It is never written in code.
if HUB_REPO:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hub checkpoints go to the PRIVATE repo", HUB_REPO, "(smoke runs use", HUB_REPO + "-smoke)")
else:
    print("No Hub repo: checkpoints stay in this session. A crash or the 12-hour limit loses the run.")

## 1. Find the extended model

In [ ]:
from pathlib import Path

# The extended model comes from the output of notebook 03, added as an input with "Add Input > Notebooks".
# The folder holds zubaan_extension.json. Prefer one that is not trained yet.
INPUT_ROOT = Path("/kaggle/input")
markers = sorted(INPUT_ROOT.rglob("zubaan_extension.json"))
assert markers, (
    "no extended model found under /kaggle/input. In the sidebar choose Add Input > Notebooks and add "
    "the finished output of 03_extend_export_qwen3"
)
untrained = [m for m in markers if not json.loads(m.read_text()).get("trained")]
marker = (untrained or markers)[0]
MODEL_DIR = marker.parent
print("model folder:", MODEL_DIR)
print(json.dumps(json.loads(marker.read_text()), indent=1))
assert (MODEL_DIR / "model.safetensors").exists() or list(MODEL_DIR.glob("model-*.safetensors")), "no weights in the model folder"
assert (MODEL_DIR / "tokenizer.json").exists(), "no tokenizer.json in the model folder"

## 2. Configuration

In [ ]:
from pathlib import Path

import yaml

WORK = Path("/kaggle/working")
adapt = {
    "model_path": str(MODEL_DIR),
    "urdu_corpus": "data/urdu/urdu_train.txt",
    "heldout": "data/urdu/urdu_heldout.txt",
    "english_corpus": "data/english/english_replay.txt",
    "english_mb": 12,
    "guard_urdu": ["data/flores_urd.txt", "data/flores_dev/flores_urd.txt"],
    "guard_english": ["data/flores_eng.txt", "data/flores_dev/flores_eng.txt"],
    "guard_belebele": True,
    "drop_overlaps": True,
    "data_dir": "data/adapt",
    "block_size": 1024,
    "english_replay_ratio": 0.1,
    "output_dir": str(WORK / OUT_NAME),
    "micro_batch_size": MICRO_BATCH,
    "grad_accum": 32 // MICRO_BATCH,
    "lr_new_rows": 1e-3,
    "lr_lora": 2e-4,
    "lora_rank": 64,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "precision": "fp16",
    "gradient_checkpointing": GRAD_CHECKPOINT,
    "device": "cuda",
    "seed": 0,
    "hub_repo": HUB,
    "hub_private": True,
    "merged_dtype": "bfloat16",
    "phase_a_steps": PHASE_A_STEPS,
    "phase_b_steps": PHASE_B_STEPS,
    "warmup_steps": 50,
    "log_every": 10,
    "eval_every": 100,
    "eval_texts": 200,
    "checkpoint_every": 50,
    "keep_checkpoints": 2,
    "milestone_every": 200,
}
config = {
    "urdu_corpus": {"raw_dir": "data/raw", "out_dir": "data/urdu", "flores": "data/flores_urd.txt", "max_mb": 100, "heldout_percent": 1.0},
    "adapt": adapt,
}
Path("adapt.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
print(Path("adapt.yaml").read_text())

## 3. Data

The same preparation as notebook 05.

In [ ]:
# Evaluation text first (FLORES devtest and dev), then the Urdu training text, then the token blocks.
# Prepare drops any training line that overlaps FLORES or Belebele text and counts it.
!zubaan data flores --out data
!zubaan data flores --split dev --out data/flores_dev
!zubaan data urdu --config adapt.yaml
!zubaan adapt prepare --config adapt.yaml

In [ ]:
meta = json.loads(Path("data/adapt/blocks_meta.json").read_text())
print("Urdu blocks", meta["urdu"]["n_blocks"], "| tokens", meta["urdu"]["n_tokens"], "| English blocks", meta["english"]["n_blocks"])
print("guard:", meta["overlap_guard"])
assert meta["overlap_guard"]["hits"] == 0
# The laptop run (2026-11-18) gave 15,677 Urdu blocks and 16,053,605 tokens. A different number means a
# different Wikipedia dump or tokenizer: not wrong by itself, but write it down.
print("laptop reference: 15677 Urdu blocks, 16053605 tokens")

## 4. Five-step fp16 check

Quick proof that this session trains before the trial starts.

In [ ]:
# CHECK 3: five real training steps in fp16, covering both phases (2 steps of new rows, 3 with LoRA).
# A non-finite loss stops the run with a message; then set precision to fp32 and rerun.
from zubaan.adapt.pipeline import run_adapt
from zubaan.config import load_config

cfg = load_config("adapt.yaml").adapt
check = cfg.model_copy(update={
    "output_dir": WORK / "adapt_check", "hub_repo": None, "resume": False,
    "phase_a_steps": 2, "phase_b_steps": 3, "grad_accum": 2, "warmup_steps": 1,
    "log_every": 1, "eval_every": 5, "eval_texts": 20, "checkpoint_every": 5,
})
run_adapt(check, log=print)
rows = [json.loads(line) for line in (check.output_dir / "train_log.jsonl").read_text().splitlines()]
losses = [r["loss"] for r in rows if "loss" in r]
# The fp16 loss scaler starts high and skips a step (gradient norm inf) when it overflows. A few at the
# start are normal; every step skipped would mean the run is not learning.
skipped = sum(1 for r in rows if "loss" in r and r["grad_norm"] == float("inf"))
print(f"steps skipped by the loss scaler: {skipped} of 5")
assert len(losses) == 5 and all(x == x and abs(x) < 1e4 for x in losses), losses
assert skipped < 5, "every step overflowed: use precision fp32"
print("OK: 5 fp16 steps, finite losses", [round(x, 3) for x in losses])

In [ ]:
import gc

import torch

gc.collect()
torch.cuda.empty_cache()
print("GPU memory in use now (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

## 5. The trial

In [ ]:
# The same few real steps under four settings, each in its own process so that an out-of-memory
# error in one cannot hurt the others. Phase B (LoRA plus new rows) is what the main run mostly does,
# so the speed is taken from phase B steps.
import copy
import shutil
import statistics
import subprocess

VARIANTS = [("mb2_ckpt_on", 2, True), ("mb4_ckpt_on", 4, True), ("mb1_ckpt_off", 1, False), ("mb2_ckpt_off", 2, False)]
base = yaml.safe_load(Path("adapt.yaml").read_text())
results = []
for name, mb, gc_on in VARIANTS:
    conf = copy.deepcopy(base)
    out = WORK / f"speed_{name}"
    shutil.rmtree(out, ignore_errors=True)
    conf["adapt"].update(
        output_dir=str(out), hub_repo=None, resume=False, micro_batch_size=mb,
        gradient_checkpointing=gc_on, grad_accum=8 // mb, phase_a_steps=2, phase_b_steps=10,
        warmup_steps=1, log_every=1, eval_every=9999, eval_texts=5, checkpoint_every=9999, keep_checkpoints=1,
    )
    cfg_file = Path(f"speed_{name}.yaml")
    cfg_file.write_text(yaml.safe_dump(conf, sort_keys=False))
    r = subprocess.run(["zubaan", "adapt", "run", "--config", str(cfg_file), "--steps", "8"], capture_output=True, text=True)
    rows = []
    if (out / "train_log.jsonl").exists():
        rows = [json.loads(line) for line in (out / "train_log.jsonl").read_text().splitlines()]
    steps = [x for x in rows if "loss" in x]
    phase_b = [x["tokens_per_s"] for x in steps if x["phase"] == "B"][1:]  # the first B step pays for the switch
    if r.returncode != 0 or len(phase_b) < 3:
        text = r.stdout + r.stderr
        why = "out of GPU memory" if "out of memory" in text.lower() else text.strip().splitlines()[-1:]
        res = {"name": name, "micro_batch": mb, "gradient_checkpointing": gc_on, "ok": False, "why": why}
    else:
        tps = statistics.median(phase_b)
        res = {
            "name": name, "micro_batch": mb, "gradient_checkpointing": gc_on, "ok": True,
            "median_tokens_per_s": round(tps), "peak_gpu_gb": round(max(x["gpu_mem_gb"] for x in steps), 2),
            "hours_per_100_steps": round(100 * 32 * 1024 / tps / 3600, 2),
        }
    print(res)
    results.append(res)
    shutil.rmtree(out, ignore_errors=True)
ok = [x for x in results if x["ok"] and x["peak_gpu_gb"] < 14.0]
assert ok, "no setting worked: send the lines above"
best = max(ok, key=lambda x: x["median_tokens_per_s"])
(WORK / "speed_report.json").write_text(json.dumps({"gpu": gpu_name, "results": results, "best": best["name"]}, indent=2))
print("fastest that fits:", best["name"], "->", best["hours_per_100_steps"], "hours per 100 steps")
print("In notebook 07 set MICRO_BATCH =", best["micro_batch"], "and GRAD_CHECKPOINT =", best["gradient_checkpointing"])
print("Send back speed_report.json.")

## 6. Hub round trip (only if HUB_REPO is set)

Uploads a stand-in checkpoint the way the main run does, checks `last/` and the milestone copy exist, downloads both, and removes the test repo.

In [ ]:
if HUB_REPO:
    import shutil

    from huggingface_hub import HfApi

    from zubaan.adapt.pipeline import download_hub_checkpoint, make_hub_uploader, milestone_path

    test_repo = HUB_REPO + "-test"
    fake = WORK / "fake_ckpt"
    shutil.rmtree(fake, ignore_errors=True)
    fake.mkdir()
    (fake / "meta.json").write_text(json.dumps({"step": 200}))
    (fake / "weights.bin").write_bytes(b"0" * 1000)
    make_hub_uploader(test_repo, True, 200)(fake, 200)
    api = HfApi()
    files = set(api.list_repo_files(test_repo))
    for need in ("last/meta.json", "last/weights.bin", "milestones/step-0000200/meta.json"):
        assert need in files, f"{need} is missing from {test_repo}: the Hub upload does not work"
    assert api.repo_info(test_repo).private, f"{test_repo} is public"
    newest = download_hub_checkpoint(test_repo, WORK / "dl_last")
    milestone = download_hub_checkpoint(test_repo, WORK / "dl_milestone", milestone_path(200))
    assert newest is not None and newest.name == "ckpt-0000200", newest
    assert milestone is not None and (milestone / "weights.bin").exists(), milestone
    try:
        api.delete_repo(test_repo)
    except Exception:
        print("could not delete", test_repo, "- delete it by hand on the Hub, it is only a test")
    print("OK: Hub upload, milestone copy and both downloads work")
else:
    print("HUB_REPO is empty: the Hub round trip was not tested. Set it to test checkpoints and milestones.")

## 7. Output size

In [ ]:
total = 0
for f in sorted(WORK.rglob("*")):
    if f.is_file() and f.stat().st_size > 5e6:
        total += f.stat().st_size
        print(f"{f.stat().st_size / 1e9:6.2f} GB  {f.relative_to(WORK)}")
print(f"total of files over 5 MB: {total / 1e9:.1f} GB")
assert total < 19e9, "output is close to Kaggle's 20 GB limit: delete data/raw or old checkpoints"